In [1]:
# 1. Parar sesión existente
from pyspark.sql import SparkSession
active = SparkSession.getActiveSession()
if active:
    active.stop()

import sys
sys.path.insert(0, '/home/jovyan/work/Data-Lakehouse')
import ingesta
# 3. Crear sesión nueva
spark = ingesta.get_spark()
print (spark.version)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-93e4603b-2bcd-474f-a197-66e206d9b759;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in spark-list
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in spark-list
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in spark-list
	found com.amazonaws#aws-java-sdk-bundle;1.12.367 in central
:: resolution rep

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
4.1.1


In [ ]:
import importlib
import pandas as pd
from IPython.display import display

import plata
importlib.reload(plata)

from plata import limpiar_tabla, validar_tabla

df = spark.table("players.mapping.teams_unified").toPandas()

print("Columnas disponibles:")
print(df.columns.tolist())
display(df.head(10))

# Rellena esta lista a mano con los grupos de columnas que quieras consolidar.
# Cada elemento debe tener:
# - col_final: nombre final de la columna consolidada
# - fuentes: columnas visibles en df que se van a unir
# - tiebreaker: columna preferida cuando haya empate
#
# Ejemplo:
# grupos_manual = [
#     {"col_final": "name", "fuentes": ["tm_name", "ts_strTeam", "fb_team"], "tiebreaker": "tm_name"},
#     {"col_final": "country", "fuentes": ["tm_country", "ts_strCountry"], "tiebreaker": "tm_country"},
# ]
grupos_manual = [
]

if not grupos_manual:
    raise ValueError("Rellena `grupos_manual` antes de ejecutar la limpieza manual.")

for indice, grupo in enumerate(grupos_manual, start=1):
    if not isinstance(grupo, dict):
        raise TypeError(f"El grupo {indice} debe ser un diccionario.")

    col_final = grupo.get("col_final")
    fuentes = grupo.get("fuentes")
    tiebreaker = grupo.get("tiebreaker")

    if not isinstance(col_final, str) or not col_final.strip():
        raise ValueError(f"El grupo {indice} necesita un `col_final` válido.")
    if not isinstance(fuentes, list) or not fuentes or not all(isinstance(c, str) and c.strip() for c in fuentes):
        raise ValueError(f"El grupo {indice} necesita una lista `fuentes` válida.")
    if not isinstance(tiebreaker, str) or not tiebreaker.strip():
        raise ValueError(f"El grupo {indice} necesita un `tiebreaker` válido.")
    if tiebreaker not in fuentes:
        raise ValueError(f"El `tiebreaker` del grupo {indice} debe estar dentro de `fuentes`.")

df_limpio = limpiar_tabla(df, grupos=grupos_manual)
df_validado, errores = validar_tabla(df_limpio)

print("\nErrores detectados:")
display(errores.head(20))

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/05/21 07:57:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Grupos detectados:
[{'columnas': ['fb_performance_int', 'tm_members', 'fb_team_success_ong', 'fb_playing_time_minpct', 'fb_performance_l', 'fb_standard_pk', 'fb_playing_time_min', 'fb_performance_crdr', 'fb_performance_sota', 'fb_performance_w', 'fb_playing_time_90s', 'fb_performance_pkcon', 'fb_players_used', 'fb_performance_tklw', 'fb_performance_pkatt', 'fb_performance_cs', 'fb_performance_d', 'fb_penalty_kicks_pksv', 'tm_stadiumSeats', 'fb_playing_time_mp', 'fb_subs_unsub', 'fb_subs_mn_sub', 'fb_starts_mn_start', 'fb_subs_subs', 'fb_performance_2crdy', 'fb_penalty_kicks_pkm', 'tm_currentTransferRecord', 'fb_penalty_kicks_savepct', 'fb_performance_off', 'fb_performance_crs', 'fb_performance_cspct', 'fb_standard_sot', 'fb_age', 'fb_per_90_minutes_ga_pk', 'fb_team_success___', 'fb_performance_ga', 'fb_performance_og', 'fb_standard_gls', 'fb_standard_pkatt', 'fb_starts_compl', 'tm_currentMarketValue', 'fb_playing_time_starts', 'fb_performance_crdy', 'fb_penalty_kicks_pkatt', 'fb_perfor